# G0 data check: well density, continuity, well types (reproducible)

Recomputes the numbers in `reports/g0_density.md` from the raw downloads.

**Inputs:**
- `data/raw/cgwb_levels/wris/` (`scripts/fetch_wris_gwl.py`)
- the India Data Portal CSV (Karnal fill)

**Outputs:** `reports/tables/g0_*.csv`

**Definitions:**
- **pre-monsoon** reading: March–June.
- **post-monsoon** reading: October–December.
- A **usable year** for a well has at least one pre-monsoon *and* one post-monsoon reading.
- **Tier 1** (zoning wells): ≥ 5 usable years within the training period 2005–2018.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data.cgwb import read_idp, read_wris
from src.utils.config import data_dir, load_config

cfg = load_config(ROOT / "configs" / "study_area.yaml")
OUT = ROOT / "reports" / "tables"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 140)

In [2]:
REGIONS = {
    "Central Punjab": ["Ludhiana", "Patiala", "Sangrur", "Fatehgarh Sahib", "Moga", "Barnala"],
    "Doaba Punjab": ["Jalandhar", "Kapurthala"],
    "Haryana belt": cfg["districts"]["Haryana"],
}
district_region = {d: r for r, ds in REGIONS.items() for d in ds}

wris = read_wris(data_dir() / "raw" / "cgwb_levels" / "wris", kind="manual")
wris["region"] = wris["district"].map(district_region)
wris["year"] = wris["time"].dt.year
wris["month"] = wris["time"].dt.month
print(f"{len(wris):,} manual readings, {wris.well_id.nunique():,} wells, "
      f"{wris.year.min()}–{wris.year.max()}")

27,460 manual readings, 1,139 wells, 1990–2024


## Well metadata (type, depth, aquifer)

In [3]:
def first_nonblank(s):
    s = s[s.astype(str).str.strip() != ""].dropna()
    return s.mode().iloc[0] if len(s) else ""

wells = wris.groupby("well_id").agg(
    district=("district", "first"), region=("region", "first"), lat=("lat", "first"), lon=("lon", "first"),
    well_type=("well_type", first_nonblank), aquifer_type=("aquifer_type", first_nonblank),
    well_depth_m=("well_depth_m", "median"), n_readings=("dtw_m", "size"),
    first_year=("year", "min"), last_year=("year", "max"))
wells["well_type"] = wells["well_type"].replace("", "unknown")
wells["aquifer_type"] = wells["aquifer_type"].replace("", "unknown")
print(wells["well_type"].value_counts().to_string(), "\n")
print(wells["aquifer_type"].value_counts().to_string(), "\n")
print("wells with depth:", wells["well_depth_m"].notna().mean().round(3))

well_type
Tube Well     733
Dug Well      261
unknown        95
Piezometer     36
Bore Well      14 

aquifer_type
Unconfined       689
unknown          342
Confined          69
Semi-Confined     39 

wells with depth: 0.916


## Usable years per well and Tier 1

In [4]:
pre_lo, pre_hi = cfg["seasons"]["pre_monsoon"]
post_lo, post_hi = cfg["seasons"]["post_monsoon"]
wris["season"] = np.select([wris.month.between(pre_lo, pre_hi), wris.month.between(post_lo, post_hi)],
                           ["pre", "post"], "other")
has = wris[wris.season != "other"].groupby(["well_id", "year"])["season"].agg(set)
usable = has[has.apply(lambda s: {"pre", "post"} <= s)].reset_index()[["well_id", "year"]]

t0, t1 = (int(cfg["period"]["train"][i][:4]) for i in (0, 1))
wells["usable_years_all"] = usable.groupby("well_id").size()
wells["usable_years_train"] = usable[usable.year.between(t0, t1)].groupby("well_id").size()
wells[["usable_years_all", "usable_years_train"]] = wells[["usable_years_all", "usable_years_train"]].fillna(0).astype(int)
wells["tier1"] = wells["usable_years_train"] >= cfg["wells"]["tier1_min_years"]

def longest_run(years):
    ys = sorted(set(years)); best = cur = 1
    for a, b in zip(ys, ys[1:]):
        cur = cur + 1 if b == a + 1 else 1
        best = max(best, cur)
    return best

wells["longest_run_years"] = wris.groupby("well_id")["year"].apply(longest_run)

region_tab = wells.groupby("region").agg(
    wells=("n_readings", "size"),
    tier1=("tier1", "sum"),
    usable_ge10_all=("usable_years_all", lambda s: (s >= 10).sum()),
    median_longest_run=("longest_run_years", "median"))
region_tab.loc["Belt total"] = [len(wells), wells.tier1.sum(), (wells.usable_years_all >= 10).sum(),
                                wells.longest_run_years.median()]
region_tab

,wells,tier1,usable_ge10_all,median_longest_run
region,,,,
Central Punjab,385.0,71.0,38.0,6.0
Doaba Punjab,158.0,40.0,25.0,7.0
Haryana belt,596.0,127.0,103.0,2.0
Belt total,1139.0,238.0,166.0,4.0


## Readings and wells per year

In [5]:
per_year = wris.groupby("year").agg(readings=("dtw_m", "size"), wells=("well_id", "nunique"))
per_year.T

year,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
readings,450,453,474,526,599,572,1154,574,562,568,...,1598,895,1301,2628,1336,711,574,1654,938,205
wells,138,139,146,164,173,172,216,190,197,188,...,617,487,640,735,472,296,293,786,406,205


## Well-type mix per year (wells reporting)
A switch from dug wells to piezometers over time would mean the network samples different depths in different years.

In [6]:
wy = wris[["well_id", "year"]].drop_duplicates().join(wells["well_type"], on="well_id")
type_mix = wy.pivot_table(index="year", columns="well_type", values="well_id", aggfunc="count", fill_value=0)
type_mix.loc[2000:]

well_type,Bore Well,Dug Well,Piezometer,Tube Well,unknown
year,,,,,
2000,0,113,0,50,0
2001,0,112,0,69,0
2002,0,108,0,71,0
2003,0,98,0,70,0
2004,0,95,0,76,0
2005,0,91,0,73,0
2006,0,93,0,71,0
2007,0,93,0,78,0
2008,0,91,0,76,0


## Depth spread

In [7]:
depth_tab = wells.groupby("well_type")["well_depth_m"].describe(percentiles=[.1, .5, .9]).round(1)
dtw_tab = wris["dtw_m"].describe(percentiles=[.1, .5, .9]).round(2)
print(dtw_tab.to_string())
depth_tab

count    27460.00
mean        16.11
std         12.43
min         -1.20
10%          2.99
50%         12.92
90%         34.10
max        250.00


,count,mean,std,min,10%,50%,90%,max
well_type,,,,,,,,
Bore Well,14.0,212.6,150.7,92.0,101.6,135.0,463.7,470.0
Dug Well,261.0,18.5,10.3,1.5,7.9,15.0,30.1,55.0
Piezometer,36.0,171.1,84.3,20.0,75.5,200.0,300.0,300.0
Tube Well,732.0,97.2,71.0,0.0,50.0,75.0,198.9,537.2
unknown,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Dry readings
WRIS has no dry-well flag. We count possible dry or problem readings instead:
- readings within 1 m of (or below) the well bottom, where depth is known
- zero or negative readings
- readings over 100 m

In [8]:
d = wris.join(wells["well_depth_m"].rename("wdepth"), on="well_id")
near_bottom = (d.wdepth.notna() & (d.dtw_m >= d.wdepth - 1)).sum()
dry_tab = pd.Series({
    "readings": len(d),
    "missing value": d.dtw_m.isna().sum(),
    "<= 0 m": (d.dtw_m <= 0).sum(),
    "> 100 m": (d.dtw_m > 100).sum(),
    "within 1 m of well bottom": near_bottom,
    "possibly dry fraction": round((near_bottom + d.dtw_m.isna().sum()) / len(d), 4),
})
dry_tab

readings                     27460.0000
missing value                    0.0000
<= 0 m                           7.0000
> 100 m                          8.0000
within 1 m of well bottom      435.0000
possibly dry fraction            0.0158
dtype: float64

## Karnal fill from the India Data Portal file

In [9]:
idp = read_idp(data_dir() / "raw" / "cgwb_levels" / "idp" / "cgwb-changes-in-depth-to-water-level.csv",
               districts={"Haryana": cfg["idp_fill_districts"]})
idp["year"] = idp.time.dt.year
karnal = pd.Series({
    "WRIS Karnal wells": (wells.district == "Karnal").sum(),
    "WRIS Karnal years": f"{wris[wris.district=='Karnal'].year.min()}–{wris[wris.district=='Karnal'].year.max()}",
    "IDP Karnal wells": idp.well_id.nunique(),
    "IDP Karnal readings": len(idp),
    "IDP Karnal years": f"{idp.year.min()}–{idp.year.max()}",
})
karnal

WRIS Karnal wells             58
WRIS Karnal years      2019–2022
IDP Karnal wells             104
IDP Karnal readings          852
IDP Karnal years       2013–2021
dtype: object

## Save tables

In [10]:
wells.to_csv(OUT / "g0_wells.csv")
region_tab.to_csv(OUT / "g0_regions.csv")
per_year.to_csv(OUT / "g0_per_year.csv")
type_mix.to_csv(OUT / "g0_type_mix.csv")
depth_tab.to_csv(OUT / "g0_depth.csv")
dry_tab.to_csv(OUT / "g0_dry.csv", header=["value"])
print("saved to", OUT)

saved to C:\Users\HP\pinn\reports\tables
